# Malaysia Tax RAG — RAGAS + Policy Evaluation

In [ ]:
import sys
import json
import re
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path("../.env"))  # repo root

# Ensure notebooks dir is on path (same as generate_testset.ipynb)
_notebooks = Path(".").resolve()
if (_notebooks / "ingest_utils.py").exists():
    sys.path.insert(0, str(_notebooks))
else:
    sys.path.insert(0, str(Path("frontend/notebooks").resolve()))

# API base — change if server runs on a different port
API_BASE = os.getenv("EVAL_API_BASE", "http://localhost:8000")
DATASET_PATH = Path(".").resolve() / "test_questions.jsonl"
TOP_K = 4  # cost control: limit retrieved chunks passed to RAGAS

print(f"API_BASE: {API_BASE}")
print(f"Dataset: {DATASET_PATH}")

API_BASE: http://localhost:8000
Dataset: /Users/victoria/Documents/AIBootcamp/Ledger/notebooks/test_questions.jsonl


## Load Golden Dataset

In [ ]:
with open(DATASET_PATH) as f:
    raw = [json.loads(l) for l in f if l.strip()]

# Assign stable IDs from concept + language style
_style_abbr = {"regulatory": "reg", "plain_user": "usr", "informal": "inf"}
_seen: dict[str, int] = {}
for row in raw:
    key = row["concept"]
    _seen[key] = _seen.get(key, 0) + 1
    style = _style_abbr.get(row["language_style"], row["language_style"][:3])
    row["id"] = f"{key[:20]}_{style}"

golden = raw
df = pd.DataFrame(golden)
print(f"Loaded {len(df)} test cases")
print(f"Retrieval types: {df['retrieval_type'].value_counts().to_dict()}")
print(f"Language styles: {df['language_style'].value_counts().to_dict()}")
print(f"Intent types:    {df['intent_type'].value_counts().to_dict()}")
try:
    display(df[["id", "question", "retrieval_type", "language_style"]].head(10))
except NameError:
    print(df[["id", "question", "retrieval_type", "language_style"]].head(10).to_string())

Loaded 16 test cases
Retrieval types: {'cross_section': 6, 'unanswerable': 6, 'answerable': 4}
Language styles: {'informal': 6, 'regulatory': 5, 'plain_user': 5}
Intent types:    {'INFO': 13, 'WHAT_IF': 3}


,id,question,retrieval_type,language_style
0,physical_presence_as_inf,I'm a Malaysian citizen living abroad — does t...,answerable,informal
1,90_day_rule_with_pri_reg,"Under paragraph 7(1)(c) of the ITA, what two c...",answerable,regulatory
2,90_day_rule_with_pri_usr,Can I qualify as a tax resident if I'm in Mala...,answerable,plain_user
3,90_day_rule_with_pri_inf,I'm only in Malaysia about 100 days a year — c...,answerable,informal
4,adjusted_loss_treatm_reg,How does the sequential step-by-step process u...,cross_section,regulatory
5,adjusted_loss_treatm_usr,"If my business makes a loss, at what stage of ...",cross_section,plain_user
6,adjusted_loss_treatm_inf,My business lost money — can I use that loss t...,cross_section,informal
7,wilful_evasion_vs_in_reg,"Under section 114(1) of the ITA 1967, what act...",cross_section,regulatory
8,wilful_evasion_vs_in_usr,What is the difference between making an hones...,cross_section,plain_user
9,wilful_evasion_vs_in_inf,If I accidentally put the wrong number and if ...,cross_section,informal


## Run Golden Dataset Against API (Eval Mode)

In [ ]:
import asyncio

async def call_eval_api(question: str, client: httpx.AsyncClient) -> dict:
    """Call /app/chat/eval and return structured response."""
    try:
        resp = await client.post(
            f"{API_BASE}/app/chat/eval",
            headers={"X-Eval-Mode": "1", "Content-Type": "application/json"},
            json={"input": question},
            timeout=60.0,
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"  ERROR for '{question[:50]}': {e}")
        return {"answer": "", "contexts": [], "policy": {}}


async def run_all(test_cases: list) -> list:
    results = []
    async with httpx.AsyncClient() as client:
        for tc in test_cases:
            print(f"  [{tc['id']}] {tc['question'][:60]}...")
            result = await call_eval_api(tc["question"], client)
            results.append({
                "id": tc["id"],
                "question": tc["question"],
                "ground_truth": tc.get("ground_truth", ""),
                "answer": result.get("answer", ""),
                "contexts": [c["text"] for c in result.get("contexts", [])[:TOP_K]],
                "contexts_full": result.get("contexts", [])[:TOP_K],
                "policy": result.get("policy", {}),
                "retrieval_type": tc.get("retrieval_type", "answerable"),
                "language_style": tc.get("language_style", ""),
                "concept": tc.get("concept", ""),
                "requires_citation": True,  # conservative: all questions require citation
            })
    return results

print("Running dataset against API...")
print(f"Total questions: {len(golden)}")
print("(Ensure server is running at", API_BASE, ")")
eval_results = await run_all(golden)
print(f"\nCollected {len(eval_results)} responses")

Running dataset against API...
Total questions: 16
(Ensure server is running at http://localhost:8000 )
  [physical_presence_as_inf] I'm a Malaysian citizen living abroad — does that automatica...
  [90_day_rule_with_pri_reg] Under paragraph 7(1)(c) of the ITA, what two cumulative cond...
  [90_day_rule_with_pri_usr] Can I qualify as a tax resident if I'm in Malaysia for less ...
  [90_day_rule_with_pri_inf] I'm only in Malaysia about 100 days a year — can I still be ...
  [adjusted_loss_treatm_reg] How does the sequential step-by-step process under section 5...
  [adjusted_loss_treatm_usr] If my business makes a loss, at what stage of calculating my...
  [adjusted_loss_treatm_inf] My business lost money — can I use that loss to reduce my re...
  [wilful_evasion_vs_in_reg] Under section 114(1) of the ITA 1967, what acts constitute w...
  [wilful_evasion_vs_in_usr] What is the difference between making an honest mistake on m...
  [wilful_evasion_vs_in_inf] If I accidentally put the wron

## Custom Policy Metrics — Deterministic (No LLM)

In [ ]:
# ── Citation keywords ──────────────────────────────────────────────────────────
_CITATION_NUMBERS_RE = re.compile(r"\b(60|182|183|30%)\b")
_CITATION_WORDS_RE = re.compile(
    r"\b(Section|PR|Article|Schedule|Form\s+BE|Form\s+B\b|Form\s+M\b)\b", re.IGNORECASE
)
_CITATION_MARKER_RE = re.compile(
    r"\b(s\d+|Section\s+\d+|PR\s+\d+/\d+|PR\s+\d+|Article\s+\d+|Schedule\s+\d+|Form\s+[BEM]E?|ITA)\b",
    re.IGNORECASE,
)
_ADVICE_LEAK_RE = re.compile(
    r"\b(you should|you must|definitely|you don't need|I recommend)\b", re.IGNORECASE
)

def one_question_compliance(answer: str) -> bool:
    cleaned = re.sub(r"\[.*?\]", "", answer)
    return cleaned.count("?") <= 1

def citation_coverage(answer: str, requires_citation: bool) -> bool:
    if not requires_citation:
        return True
    has_trigger = bool(_CITATION_NUMBERS_RE.search(answer) or _CITATION_WORDS_RE.search(answer))
    if not has_trigger:
        return True
    return bool(_CITATION_MARKER_RE.search(answer))

def advice_leakage(answer: str) -> bool:
    return not bool(_ADVICE_LEAK_RE.search(answer))


# ── Apply policy metrics (all cases incl. unanswerable) ───────────────────────
policy_rows = []
for r in eval_results:
    ans = r["answer"]
    policy_rows.append({
        "id": r["id"],
        "retrieval_type": r["retrieval_type"],
        "language_style": r["language_style"],
        "one_question_compliance": one_question_compliance(ans),
        "citation_coverage": citation_coverage(ans, r["requires_citation"]),
        "advice_leakage_pass": advice_leakage(ans),
        "num_q_marks": ans.count("?"),
        "num_contexts": len(r["contexts"]),
        "max_score": max((c.get("score", 0) for c in r["contexts_full"]), default=0.0),
    })

policy_df = pd.DataFrame(policy_rows)

# Restrict policy summary to answerable + cross_section (unanswerable skews citation metrics)
scoreable = policy_df[policy_df["retrieval_type"] != "unanswerable"]

oqc_pct     = scoreable["one_question_compliance"].mean() * 100
cc_pct      = scoreable["citation_coverage"].mean() * 100
al_pct      = policy_df["advice_leakage_pass"].mean() * 100
avg_chunks  = scoreable["num_contexts"].mean()
avg_max_score = scoreable["max_score"].mean()

print(f"Scoreable cases (answerable + cross_section): {len(scoreable)}")
print(f"Unanswerable cases (excluded from recall/p\ecision): {len(policy_df) - len(scoreable)}")
print(f"\nOne Question Compliance:  {oqc_pct:.1f}%")
print(f"Citation Coverage:        {cc_pct:.1f}%")
print(f"Advice Leakage (pass%):   {al_pct:.1f}%")
print(f"Avg chunks per question:  {avg_chunks:.2f}")
print(f"Avg max similarity score: {avg_max_score:.4f}")

# Breakdown by language style
print("\n── By language style (scoreable) ──")
print(scoreable.groupby("language_style")[["one_question_compliance","citation_coverage"]].mean().round(3).to_string())

Scoreable cases (answerable + cross_section): 10
Unanswerable cases (excluded from recall/p\ecision): 6

One Question Compliance:  100.0%
Citation Coverage:        90.0%
Advice Leakage (pass%):   100.0%
Avg chunks per question:  3.60
Avg max similarity score: 0.7510

── By language style (scoreable) ──
                one_question_compliance  citation_coverage
language_style                                            
informal                            1.0               0.75
plain_user                          1.0               1.00
regulatory                          1.0               1.00


<>:58: SyntaxWarning: invalid escape sequence '\e'
<>:58: SyntaxWarning: invalid escape sequence '\e'
/var/folders/l9/k4g359vd6vg5k697lr_4l69c0000gn/T/ipykernel_94841/593119710.py:58: SyntaxWarning: invalid escape sequence '\e'
  print(f"Unanswerable cases (excluded from recall/p\ecision): {len(policy_df) - len(scoreable)}")


## RAGAS Evaluation (LLM-Assisted)

In [ ]:
import warnings
import instructor
from datasets import Dataset
from ragas import evaluate
from ragas.dataset_schema import EvaluationResult
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerRelevancy
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings as LCOpenAIEmbeddings
from ragas.llms import LiteLLMStructuredLLM

# Filter: exclude unanswerable (no ground truth) and cases without answer+contexts
ragas_rows = [
    r for r in eval_results
    if r["retrieval_type"] != "unanswerable"
    and r["answer"]
    and r["contexts"]
    and not r["ground_truth"].startswith("Insufficient information")
]
print(f"Cases for RAGAS: {len(ragas_rows)} "
      f"(skipped {len(eval_results) - len(ragas_rows)} unanswerable/empty)")

# Breakdown by retrieval type
from collections import Counter
print("Breakdown:", Counter(r["retrieval_type"] for r in ragas_rows))

ragas_data = {
    "question":     [r["question"] for r in ragas_rows],
    "answer":       [r["answer"] for r in ragas_rows],
    "contexts":     [r["contexts"] for r in ragas_rows],
    "ground_truth": [r["ground_truth"] for r in ragas_rows],
}
ragas_dataset = Dataset.from_dict(ragas_data)

_oai_instructor = instructor.from_openai(OpenAI())
evaluator_llm = LiteLLMStructuredLLM(client=_oai_instructor, model="gpt-4o-mini", provider="openai")
evaluator_embeddings = LCOpenAIEmbeddings(model="text-embedding-3-small")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning)
    faithfulness_m   = Faithfulness();      faithfulness_m.llm   = evaluator_llm
    context_prec_m   = ContextPrecision();  context_prec_m.llm   = evaluator_llm
    context_rec_m    = ContextRecall();     context_rec_m.llm    = evaluator_llm
    answer_rel_m     = AnswerRelevancy(strictness=1); answer_rel_m.llm = evaluator_llm
    answer_rel_m.embeddings = evaluator_embeddings
    metrics = [faithfulness_m, context_prec_m, context_rec_m, answer_rel_m]

ragas_result = evaluate(ragas_dataset, metrics=metrics)
assert isinstance(ragas_result, EvaluationResult), f"Unexpected return type: {type(ragas_result)}"

ragas_scores: pd.DataFrame = ragas_result.to_pandas()
metric_cols = [c for c in ["faithfulness", "context_precision", "context_recall", "answer_relevancy"] if c in ragas_scores.columns]

# Add metadata back for breakdown analysis
for col, vals in [("retrieval_type", [r["retrieval_type"] for r in ragas_rows]),
                  ("language_style",  [r["language_style"]  for r in ragas_rows])]:
    ragas_scores[col] = vals

print(f"\nRAGAS per-case scores ({len(ragas_scores)} cases)")
try:
    display(ragas_scores[["user_input"] + metric_cols].head(20))
except NameError:
    print(ragas_scores[["user_input"] + metric_cols].head(20).to_string())

# Breakdown by language style
print("\n── Context Recall by language style ──")
print(ragas_scores.groupby("language_style")["context_recall"].mean().round(3).to_string())

/var/folders/l9/k4g359vd6vg5k697lr_4l69c0000gn/T/ipykernel_94841/1538469262.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerRelevancy
/var/folders/l9/k4g359vd6vg5k697lr_4l69c0000gn/T/ipykernel_94841/1538469262.py:6: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerRelevancy
/var/folders/l9/k4g359vd6vg5k697lr_4l69c0000gn/T/ipykernel_94841/1538469262.py:6: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.

Cases for RAGAS: 9 (skipped 7 unanswerable/empty)
Breakdown: Counter({'cross_section': 5, 'answerable': 4})


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]


RAGAS per-case scores (9 cases)


,user_input,faithfulness,context_precision,context_recall,answer_relevancy
0,I'm a Malaysian citizen living abroad — does t...,0.857143,1.000000,1.000000,0.914712
1,"Under paragraph 7(1)(c) of the ITA, what two c...",0.500000,0.638889,1.000000,0.410354
2,Can I qualify as a tax resident if I'm in Mala...,0.785714,1.000000,1.000000,0.000000
3,I'm only in Malaysia about 100 days a year — c...,0.666667,1.000000,0.500000,0.000000
4,How does the sequential step-by-step process u...,0.562500,1.000000,1.000000,0.000000
5,"If my business makes a loss, at what stage of ...",0.615385,1.000000,0.666667,0.672091
6,"Under section 114(1) of the ITA 1967, what act...",0.958333,0.916667,0.250000,0.860420
7,What is the difference between making an hones...,0.250000,1.000000,0.500000,0.705929
8,If I accidentally put the wrong number and if ...,0.619048,1.000000,0.000000,0.579984



── Context Recall by language style ──
language_style
informal      0.500
plain_user    0.722
regulatory    0.750


## Results Table

In [6]:
def safe_mean(series):
    try:
        return series.dropna().mean()
    except Exception:
        return float("nan")

faithfulness_score = safe_mean(ragas_scores["faithfulness"])       if "faithfulness"       in ragas_scores.columns else float("nan")
cp_score           = safe_mean(ragas_scores["context_precision"])  if "context_precision"  in ragas_scores.columns else float("nan")
cr_score           = safe_mean(ragas_scores["context_recall"])     if "context_recall"     in ragas_scores.columns else float("nan")
ar_score           = safe_mean(ragas_scores["answer_relevancy"])   if "answer_relevancy"   in ragas_scores.columns else float("nan")

results_table = pd.DataFrame([
    {"Metric": "Faithfulness",             "Score": f"{faithfulness_score:.2f}"},
    {"Metric": "Context Precision",        "Score": f"{cp_score:.2f}"},
    {"Metric": "Context Recall",           "Score": f"{cr_score:.2f}"},
    {"Metric": "Answer Relevancy",         "Score": f"{ar_score:.2f}"},
    {"Metric": "One Question Compliance",  "Score": f"{oqc_pct:.1f}%"},
    {"Metric": "Citation Coverage",        "Score": f"{cc_pct:.1f}%"},
    {"Metric": "Advice Leakage (pass%)",   "Score": f"{al_pct:.1f}%"},
])

print("=" * 40)
print("EVALUATION RESULTS SUMMARY")
print("=" * 40)
print(f"Total cases evaluated:        {len(eval_results)}")
print(f"Cases with RAGAS scores:      {len(ragas_rows)}")
print(f"Avg retrieved chunks/Q:       {avg_chunks:.2f}")
print(f"Avg max similarity score:     {avg_max_score:.4f}")
print()
try:
    display(results_table.set_index("Metric"))
except NameError:
    print(results_table.to_string(index=False))

# Diagnostic summary
print("\n── Diagnostic Notes ──")
low_oqc = policy_df[~policy_df["one_question_compliance"]]
if not low_oqc.empty:
    print(f"OQC failures ({len(low_oqc)}): {low_oqc['id'].tolist()}")
else:
    print("OQC: all passed")

low_cc = policy_df[~policy_df["citation_coverage"]]
if not low_cc.empty:
    print(f"Citation coverage failures ({len(low_cc)}): {low_cc['id'].tolist()}")
else:
    print("Citation coverage: all passed")

low_al = policy_df[~policy_df["advice_leakage_pass"]]
if not low_al.empty:
    print(f"Advice leakage detected ({len(low_al)}): {low_al['id'].tolist()}")
else:
    print("Advice leakage: none detected")

if "faithfulness" in ragas_scores.columns:
    low_faith_cases = ragas_scores[ragas_scores["faithfulness"] < 0.5]
    if not low_faith_cases.empty:
        print(f"Low faithfulness (<0.5): {[s[:50] for s in low_faith_cases['user_input'].astype(str).tolist()]}")

EVALUATION RESULTS SUMMARY
Total cases evaluated:        16
Cases with RAGAS scores:      9
Avg retrieved chunks/Q:       3.60
Avg max similarity score:     0.7510



,Score
Metric,
Faithfulness,0.65
Context Precision,0.95
Context Recall,0.66
Answer Relevancy,0.46
One Question Compliance,100.0%
Citation Coverage,90.0%
Advice Leakage (pass%),100.0%



── Diagnostic Notes ──
OQC: all passed
Citation coverage failures (3): ['physical_presence_as_inf', 'advance_ruling_appli_usr', 'advance_ruling_appli_inf']
Advice leakage: none detected
Low faithfulness (<0.5): ['What is the difference between making an honest mi']


## Save Results

In [7]:
out_dir = Path(".").resolve()

# Save eval_results.json (full per-case results)
eval_output = []
for i, r in enumerate(eval_results):
    row = {
        "id": r["id"],
        "question": r["question"],
        "answer": r["answer"],
        "contexts": r["contexts"],
        "ground_truth": r["ground_truth"],
        "policy": r["policy"],
        "type": r["type"],
    }
    # Add RAGAS scores if available
    if i < len(ragas_scores):
        for col in ["faithfulness", "context_precision", "context_recall", "answer_relevancy"]:
            val = ragas_scores.iloc[i].get(col)
            if val is not None and not pd.isna(val):
                row[col] = float(val)
    eval_output.append(row)

eval_path = out_dir / "eval_results.json"
with open(eval_path, "w") as f:
    json.dump(eval_output, f, indent=2)
print(f"Saved eval_results.json → {eval_path}")

# Print final summary for record
print("\nFinal scores saved. Run complete.")

KeyError: 'type'